In [1]:
import pandas as pd
import numpy as np
from scipy.stats import poisson
import statsmodels.api as sm  # Added missing import
import statsmodels.formula.api as smf
from sklearn.linear_model import LogisticRegression

# 1. Load Data
# Ensure your file paths are correct for your environment
train = pd.read_csv("/kaggle/input/competitions/wc2026-ai-prediction/train.csv")
test = pd.read_csv("/kaggle/input/competitions/wc2026-ai-prediction/test_wc2022.csv")
train['date'] = pd.to_datetime(train['date'])

# 2. Time-Weighted Data (Focusing on the modern era)
max_date = train['date'].max()
train['days_ago'] = (max_date - train['date']).dt.days
# Matches from ~10 years ago have about 50% the weight of today
train['weight'] = np.exp(-0.0002 * train['days_ago']) 

# 3. ENGINE 1: Weighted Poisson (Goal Dynamics)
goal_data = pd.concat([
    train[['home_team', 'away_team', 'home_score', 'weight']].rename(
        columns={'home_team':'t', 'away_team':'o', 'home_score':'g'}),
    train[['away_team', 'home_team', 'away_score', 'weight']].rename(
        columns={'away_team':'t', 'home_team':'o', 'away_score':'g'})
])

# Use WLS logic via weights in GLM
# This model calculates Attack (t) and Defense (o) strengths
model_poisson = smf.glm(formula="g ~ t + o", data=goal_data, 
                        family=sm.families.Poisson(), 
                        freq_weights=goal_data['weight']).fit()

# 4. ENGINE 2: Optimized Elo (General Prestige)
def get_optimized_elo(df):
    teams = pd.concat([df['home_team'], df['away_team']]).unique()
    elo = {t: 1500 for t in teams}
    for _, row in df.iterrows():
        h, a = row['home_team'], row['away_team']
        # Goal margin multiplier (Diminishing returns for blowouts)
        margin = 1 + np.log(abs(row['home_score'] - row['away_score']) + 1)
        exp_h = 1 / (1 + 10**((elo[a] - elo[h])/400))
        act_h = 1 if row['home_score'] > row['away_score'] else (0.5 if row['home_score'] == row['away_score'] else 0)
        
        # High K for World Cup, lower for friendlies
        k = 50 if "World Cup" in str(row['tournament']) else 30
        delta = k * margin * (act_h - exp_h)
        elo[h] += delta
        elo[a] -= delta
    return elo

elo_map = get_optimized_elo(train)

# 5. DIXON-COLES RHO CORRECTION
# Corrects for the fact that 0-0 and 1-1 happen more often than random Poisson suggests
def rho_correction(x, y, mu, lam, rho):
    if x == 0 and y == 0: return 1 - (mu * lam * rho)
    if x == 0 and y == 1: return 1 + (mu * rho)
    if x == 1 and y == 0: return 1 + (lam * rho)
    if x == 1 and y == 1: return 1 - rho
    return 1

def get_titan_probs(h_team, a_team, rho=-0.05):
    try:
        # Predict lambda for both teams
        mu = model_poisson.predict(pd.DataFrame({'t': [h_team], 'o': [a_team]}))[0]
        lam = model_poisson.predict(pd.DataFrame({'t': [a_team], 'o': [h_team]}))[0]
        
        # Build score matrix
        max_g = 9
        m = np.zeros((max_g, max_g))
        for x in range(max_g):
            for y in range(max_g):
                m[x, y] = poisson.pmf(x, mu) * poisson.pmf(y, lam) * rho_correction(x, y, mu, lam, rho)
        
        m = np.clip(m, 0, None)
        m /= m.sum()
        p_pois = np.array([np.sum(np.tril(m, -1)), np.sum(np.diag(m)), np.sum(np.triu(m, 1))])
    except Exception:
        # Fallback if a team is missing from training data (like Qatar in older sets)
        p_pois = np.array([0.334, 0.332, 0.334])

    # Elo Diff Logic
    e_h = elo_map.get(h_team, 1500)
    e_a = elo_map.get(a_team, 1500)
    diff = e_h - e_a
    
    # Sigmoid Win Probability
    win_p = 1 / (1 + 10**(-diff/450))
    # Draw Probability (Grounded in ELO parity)
    draw_p = 0.29 * np.exp(-abs(diff)/750)
    
    p_elo = np.array([win_p*(1-draw_p), draw_p, (1-win_p)*(1-draw_p)])

    # Weighted Ensemble (0.75 Poisson / 0.25 Elo)
    final = (p_pois * 0.75) + (p_elo * 0.25)
    
    # LEADERBOARD PROTECTOR: Ensure no result is < 5% or > 82%
    final = np.clip(final, 0.05, 0.82)
    return final / final.sum()

# 6. Generate and Format
print("Calculating Titan Ensemble Predictions...")
results = []
for _, row in test.iterrows():
    results.append(get_titan_probs(row['home_team'], row['away_team']))

preds = np.array(results)

# 7. FINAL LOG-LOSS SMOOTHING (The "Entropy Nudge")
# Blending with 10% uniform distribution protects against massive upsets
preds = (preds * 0.90) + (0.10 * 1/3)

submission = pd.DataFrame({
    'match_id': test['match_id'],
    'p_home_win': preds[:, 0],
    'p_draw': preds[:, 1],
    'p_away_win': preds[:, 2]
})

submission.to_csv("submission.csv", index=False)
print("✅ Submission.csv generated successfully!")

Calculating Titan Ensemble Predictions...
✅ Submission.csv generated successfully!
